# Multicollinearity and Influential Points

In this notebook we will identify multicollinearity in a set of predictors and demonstrate how it causes issues with modeling. We will also show how we can identify outliers, or **influential** points. We'll be using the **Credit** data for this exercise.

In [ ]:
# Detect Multicollinearity 
import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf


In [ ]:
# 1. We want to create a heatmap for pairwise correlations

credit = pd.read_csv("../Data/Credit.csv")
credit['Income'] = pd.to_numeric(credit['Income'])
credit.sample(5)

In [ ]:
# We need the numeric variables for the correlation matrix
quantitative_vars = [ 'Balance', 'Income', 'Limit', 'Rating', 'Cards', 'Age']

# Calculate the correlation matrix
corr_matrix = credit[quantitative_vars].corr()
corr_matrix

In [ ]:
# For visulaization purposes, to spot highly correlated predictors, we can use a heatmap.
import matplotlib.colors as mcolors

# Create a custom diverging colormap where 0 is explicitly white
cmap = mcolors.LinearSegmentedColormap.from_list('custom_coolwarm', ['red', 'white', 'blue'], N=256)

# Create a plot to visualize the correlation matrix with white for zero
fig, ax = plt.subplots(figsize=(8, 8))

# Apply the custom colormap to the heatmap
cax = ax.matshow(corr_matrix, cmap=cmap, vmin=-1, vmax=1)

# Add color bar to the side
plt.colorbar(cax)

# Set the ticks for the x and y axis
ax.set_xticks(range(len(quantitative_vars)))
ax.set_yticks(range(len(quantitative_vars)))

# Set the labels for the ticks
ax.set_xticklabels(quantitative_vars, rotation=45, ha='left')
ax.set_yticklabels(quantitative_vars)

# Annotate the correlation matrix with the numeric values
for (i, j), val in np.ndenumerate(corr_matrix):
    ax.text(j, i, f'{val:.2f}', ha='center', va='center', color='black')

# Display the plot
plt.title("Correlation Matrix")
plt.show()


### Note:

- We are only comparing quantitative predictors here, not categorical. 
- Think: how can we measure the "correlation" between categorical variables? Or categorical variables vs. quantitative variables? It is not straightforward.

### Problem with identifying with correlations:
- It characterizes pairwise relationships, but does not evaluate all features' joint correlations together.
- It doesn't make sense out of the box for categorical variables. 

We can use the variance inflation factors to identify predictors that are linear combinations of more than one other predictor since the correlation is only a pairwise metric.

In [ ]:
# VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices #an alternative to basic pandas dfs.

y, X = dmatrices('Balance ~ Income + Limit + Rating + Cards + \
                  Age + Education + Gender + Student + Married + Ethnicity', 
                  data=credit, return_type='dataframe')

vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif["features"] = X.columns
print(vif)

Limit and Rating are identified as variables that are causing the multicolinearity problem. Since they are highly correlated, we can consider deleting one. Let's try to delete Limit.

In [ ]:
# Deleting limit improves VIFs considerably
y, X_new = dmatrices('Balance~ Income + Rating + Cards + \
                      Age + Education + Gender + Student + Married + Ethnicity', 
                      data=credit, return_type='dataframe')
vif = pd.DataFrame()
vif["VIF Factor"] = [variance_inflation_factor(X_new.values, i) for i in range(X_new.shape[1])]
vif["features"] = X_new.columns
print(vif)

# The intercept has a high VIF but this doesn't mean we should necessarily get rid of it. 
# If we do, we are assuming true proportionality between X and y. That's a big assumption. 
# Especially since from the summary table that it's not close to zero. 
# Btw - what are the null and alt hypotheses for that t - test? Do you remember? 

In [ ]:
# First, fit model with Limit AND rating 
model = smf.ols('Balance ~ Income + Limit + Rating + Cards + \
                Age + Education + Gender + Student + Married + Ethnicity', data=credit).fit()
model.summary()

In [ ]:
# Now fit new model without limit, since it is highly correlated with rating.
model = smf.ols('Balance ~ Income + Rating + Cards + \
                Age + Education + Gender + Student + Married + Ethnicity', data=credit).fit()
model.summary()

How different is the model above when we removed one of the highly correlated predictors? Take a look at the p-values for each coefficient, and look at the coefficients themselves. 

#### Perfect Collinearity

Let's create a feature that is perfectly correlated with one of our others and see what happens. You'll notice that the model will still be fit. The statsmodels library can still compute the inverse of the $X^TX$ matrix using a different method, but look at the very bottom of the summary, in the fine print. You'll see a *warning* about your matrix, which tells you not to fully trust the results.

In [ ]:
# Create a new feature that is a linear combination of the two highly correlated features.
credit['Limit_Rating'] = credit['Limit']*2

# Fit a model with the new feature
model = smf.ols('Balance ~ Income + Rating + Limit + Limit_Rating + Cards + \
                Age + Education + Gender + Student + Married + Ethnicity', data=credit).fit()
model.summary()

### Simulated Example

In this section let's simulate some data ourselves so that we can control the amount of correlation.

In [ ]:
# Uncorrelated predictor variables
X1 = [4, 4, 4, 4, 6, 6, 6, 6]
X2 = [2, 2, 3, 3, 2, 2, 3, 3]
Y = [42, 39, 48, 51, 49, 53, 61, 60]

df = pd.DataFrame({
    'X1': X1,
    'X2': X2,
    'Y': Y
})

# Plot a scatterplot of X2 against X1
plt.scatter(df['X1'], df['X2'])

In [ ]:
# Let's check that there is no correlation between X1 and X2
correlation = np.corrcoef(X1, X2)[0, 1]
print(f'Correlation between X1 and X2: {correlation}')

Now let's fit three models and compare the coefficients.

In [ ]:
# Linear model Y ~ X1 + X2
# Let's check coefficients for the model
model1 = smf.ols('Y ~ X1 + X2', data=df).fit()
model1.params

In [ ]:
model2 = smf.ols('Y ~ X1', data=df).fit()
model2.params

In [ ]:
model3 = smf.ols('Y ~ X2', data=df).fit()
model3.params

**Question:** What did you notice about the coefficients?

#### Now let's simulate a perfectly correlated dataset

In [ ]:
# Perfectly correlated case
X1 = [2, 8, 6, 10]
X2 = [6, 9, 8, 10]
Y = [23, 83, 63, 103]

df_perfect = pd.DataFrame({
    'X1': X1,
    'X2': X2,
    'Y': Y
})

# Plot a scatterplot of X2 against X1
plt.scatter(df_perfect['X1'], df_perfect['X2'])

In [ ]:
correlation_perfect = np.corrcoef(X1, X2)[0, 1]
print(f'Correlation between X1 and X2: {correlation_perfect}')

Let's create the design matrix and do some of the linear algebra ourselves.

In [ ]:
X = np.column_stack([np.ones(len(X1)), X1, X2])
X

In [ ]:
# Matrix operations

XtX = np.dot(X.T, X)

# Look at the eigenvalues of XtX to see if it is singular or near singular
eigenvalues = np.linalg.eigvals(XtX)
print(f'Eigenvalues:\n{eigenvalues}') # 2nd eigen val very close to zero !!!! 
# numerically near singular

In [ ]:
import scipy.linalg as spla

# Attempt to invert the matrix
XtX_inv = spla.solve(XtX, np.eye(3))
print(f'Inverse of XtX:\n{XtX_inv}')

Notice in the above matrix how large the values are. Also note the warning message! We can check if the inverse is correct easily.

In [ ]:
# Multiply the inverse by the original matrix to check if we get the identity matrix
XtX_inv @ XtX

In [ ]:
# Let's use np.linalg.inv to invert the matrix, but be cautious as it may not handle singular matrices well.
# It doesn't even give you a warning!!
np.linalg.inv(XtX)

Let's do the same thing we did above, let's fit the three models and compare coefficients.

In [ ]:
# Linear model Y ~ X1 + X2
# Let's check coefficients for the model
model1 = smf.ols('Y ~ X1 + X2', data=df_perfect).fit()
model1.summary()

In [ ]:
model2 = smf.ols('Y ~ X1', data=df_perfect).fit()
model2.summary()

In [ ]:
model3 = smf.ols('Y ~ X2', data=df_perfect).fit()
model3.summary()

## VIF

Let's take a look at how we can calculate VIF for our predictors.

In [ ]:
# Variance Inflation - generating new data
np.random.seed(0)
X1 = np.random.uniform(25, 50, 100)
X2 = np.random.normal(10, 3, 100)
X3 = 2 * X1 + 3 * X2 + np.random.normal(0, 0.01, 100)
eps = np.random.normal(0, 4, 100)

df_vif = pd.DataFrame({
    'X1': X1,
    'X2': X2,
    'X3': X3,
    'Y': 7 + 3 * X1 + 5 * X2 + eps
})

vif_model_X1o = smf.ols('X2 ~ X1', data=df_vif).fit()
r2j_X1o = vif_model_X1o.rsquared
vif_j_x1 = 1/(1-r2j_X1o)
print(f'VIF for X1: {vif_j_x1}')

In the code above we showed an example of how to calculate the VIF manually for `X2`, but we didn't include all of the predictors. We can see that `X2` and `X1` are definitely not correlated. However, include all of the other predictors and see what happens to the VIF.